<a href="https://colab.research.google.com/github/shraddhabhongade29-prog/ML105/blob/main/Gaussian_Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 8: Gaussian Naive Bayes Classification

**Aim:** Implement Gaussian Naive Bayes manually and using Scikit-learn to classify sample email messages as Spam or Ham.


**Libraries:** NumPy, Pandas, Scikit-learn

**Note:** Each email is represented by three numeric word-count features: `free_count`, `offer_count`, and `meeting_count`. This is a small educational dataset.

## 1. Import libraries and create the dataset

In [9]:
import numpy as np
import pandas as pd

data = pd.DataFrame({
    "free_count": [2, 4, 2, 1, 0, 1, 0, 1, 3, 0, 2, 0],
    "offer_count": [1, 1, 1, 0, 0, 0, 0, 0, 2, 0, 1, 0],
    "meeting_count": [0, 0, 0, 3, 2, 2, 4, 3, 0, 2, 0, 3],
    "label": ["Spam", "Spam", "Spam", "Ham", "Ham", "Ham",
              "Ham", "Ham", "Spam", "Ham", "Spam", "Ham"]
})

print("Email Dataset:")
print(data)

Email Dataset:
    free_count  offer_count  meeting_count label
0            2            1              0  Spam
1            4            1              0  Spam
2            2            1              0  Spam
3            1            0              3   Ham
4            0            0              2   Ham
5            1            0              2   Ham
6            0            0              4   Ham
7            1            0              3   Ham
8            3            2              0  Spam
9            0            0              2   Ham
10           2            1              0  Spam
11           0            0              3   Ham


## 2. Separate features and labels, then split the data

In [2]:
X = data[["free_count", "offer_count", "meeting_count"]].to_numpy(dtype=float)
y = data["label"].to_numpy()

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Classes:", np.unique(y))

Training samples: 9
Testing samples: 3
Classes: ['Ham' 'Spam']


## 3. Implement Gaussian Naive Bayes manually

The Gaussian model calculates the mean and variance of each feature for each class. The implementation below uses the **log Gaussian probability** directly. This avoids array comparisons with Python's built-in `max()` and is more numerically stable.

In [3]:
def train_gaussian_nb(X, y):
    model = {}
    for cls in np.unique(y):
        X_class = X[y == cls]
        model[cls] = {
            "prior": len(X_class) / len(X),
            "mean": np.mean(X_class, axis=0),
            "variance": np.maximum(np.var(X_class, axis=0), 1e-9)
        }
    return model


def predict_one(model, sample):
    class_scores = {}

    for cls, values in model.items():
        mean = values["mean"]
        variance = values["variance"]

        # Log of Gaussian probability for each feature
        log_probability = -0.5 * np.log(2 * np.pi * variance)
        log_probability -= ((sample - mean) ** 2) / (2 * variance)

        # Add log prior and sum feature log probabilities
        score = np.log(values["prior"]) + np.sum(log_probability)
        class_scores[cls] = float(score)

    return max(class_scores, key=class_scores.get)


manual_model = train_gaussian_nb(X_train, y_train)
manual_predictions = np.array([
    predict_one(manual_model, sample) for sample in X_test
])

print("Manual Gaussian Naive Bayes predictions:")
print(manual_predictions)

Manual Gaussian Naive Bayes predictions:
['Spam' 'Ham' 'Ham']


## 4. Evaluate the manual model

In [10]:
from sklearn.metrics import accuracy_score, classification_report

manual_accuracy = accuracy_score(y_test, manual_predictions)

print("Actual labels:   ", y_test)
print("Predicted labels:", manual_predictions)
print("Manual model accuracy:", round(manual_accuracy, 2))
print("\nClassification Report:")
print(classification_report(y_test, manual_predictions, zero_division=0))

Actual labels:    ['Spam' 'Ham' 'Ham']
Predicted labels: ['Spam' 'Ham' 'Ham']
Manual model accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         Ham       1.00      1.00      1.00         2
        Spam       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



## 5. Predict a new email using the manual model

In [11]:
# Example email: 1 occurrences of 'free', 3 of 'offer', and 0 of 'meeting'
new_email = np.array([1, 3, 0], dtype=float)

new_prediction = predict_one(manual_model, new_email)

print("New email features:", new_email)
print("New email classification:", new_prediction)

New email features: [1. 3. 0.]
New email classification: Spam


## 6. Implement Gaussian Naive Bayes using Scikit-learn

In [6]:
from sklearn.naive_bayes import GaussianNB

classifier = GaussianNB()
classifier.fit(X_train, y_train)

sklearn_predictions = classifier.predict(X_test)
sklearn_accuracy = accuracy_score(y_test, sklearn_predictions)

print("Actual labels:   ", y_test)
print("Predicted labels:", sklearn_predictions)
print("Scikit-learn accuracy:", round(sklearn_accuracy, 4))
print("\nClassification Report:")
print(classification_report(y_test, sklearn_predictions, zero_division=0))

Actual labels:    ['Spam' 'Ham' 'Ham']
Predicted labels: ['Spam' 'Ham' 'Ham']
Scikit-learn accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         Ham       1.00      1.00      1.00         2
        Spam       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



## 7. Compare manual and Scikit-learn implementations

In [12]:
comparison = pd.DataFrame({
    "Actual": y_test,
    "Manual Prediction": manual_predictions,
    "Scikit-learn Prediction": sklearn_predictions
})

print(comparison)
print("\nManual accuracy:", round(manual_accuracy, 5))
print("Scikit-learn accuracy:", round(sklearn_accuracy, 5))

  Actual Manual Prediction Scikit-learn Prediction
0   Spam              Spam                    Spam
1    Ham               Ham                     Ham
2    Ham               Ham                     Ham

Manual accuracy: 1.0
Scikit-learn accuracy: 1.0


## Observations

1. Gaussian Naive Bayes is a supervised learning algorithm based on Bayes' theorem.
2. It assumes that each feature follows a Gaussian distribution within each class.
3. The manual implementation calculates class prior probabilities, means, and variances.
4. Log probabilities are used to improve numerical stability.
5. Scikit-learn provides a ready-to-use `GaussianNB` classifier.
6. The model classifies sample emails as Spam or Ham.
7. The two implementations are compared using test-set accuracy.

## Result

Gaussian Naive Bayes was implemented manually and using Scikit-learn. Sample email messages were classified into Spam and Ham categories, and both implementations were evaluated using a held-out test set.